# NB07 — Statistical Analysis

Pure stats on saved artifacts (no training / no GPU). Upstream:
- **NB03** Stage-1 metrics + Tufts zero-shot preds
- **NB04 Part 1** frozen CV report + Tufts Stage-2 / OralXray forgetting metrics
- **NB05** refit Tufts preds + DENTEX Stage-2 / Stage-1 preds

**Tests (secondary confirmatory — plan §7.1–7.3):**
1. **Forgetting** — OralXrays Dice drop Stage-1 → Stage-2
2. **McNemar** — Tufts healthy subset, Stage-2 vs Stage-1 (pilot; underpowered)
3. **DENTEX OOD** — AUROC + paired bootstrap dAUROC (primary OOD claim). Accuracy / Wilcoxon at thresholds are diagnostics only (DENTEX is ~93% positive; shipped thresholds can be degenerate).

Primary detection claim remains **NB04 CV + bootstrap**. Do not overclaim “gaze wins” from NB07 alone.

**Outputs:** `statistics/mcnemar_table.csv`, `statistics/p_values.json`, `statistics/statistics_status.json`


In [2]:
# === Colab setup — local fallback so NB07 also runs off-Colab ===
import os, sys, json

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

PROJECT = '/content/drive/MyDrive/DENTAL'
if not os.path.isfile(os.path.join(PROJECT, 'config.json')):
    for cand in (os.getcwd(), os.path.abspath('..')):
        if os.path.isfile(os.path.join(cand, 'config.json')):
            PROJECT = os.path.abspath(cand)
            break

NOTEBOOKS_DIR = os.path.join(PROJECT, 'Notebooks')
sys.path = [p for p in sys.path if p not in (PROJECT, NOTEBOOKS_DIR)]
sys.path.insert(0, PROJECT)
sys.path.insert(0, NOTEBOOKS_DIR)
os.chdir(PROJECT)

with open('config.json', 'r') as f:
    CONFIG = json.load(f)

print('Working directory:', os.getcwd())


Mounted at /content/drive
Working directory: /content/drive/MyDrive/DENTAL


In [3]:
# === Deps (Colab + local) ===
import sys, subprocess
pkgs = ['scipy', 'statsmodels', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'packaging']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('Deps OK')


Deps OK


## 1 — Runtime setup

No GPU required. Real tests run whenever NB03/NB04/NB05 artifacts are present.


In [4]:
import numpy as np
import pandas as pd
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ON_GPU = torch.cuda.is_available()

os.makedirs('statistics', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f'ON_GPU={ON_GPU} (not required) | device={device}')
print('Real tests run when upstream artifacts are present; otherwise synthetic preflight only.')


ON_GPU=False (not required) | device=cpu
Real tests run when upstream artifacts are present; otherwise synthetic preflight only.


## 2 — Input file inventory

Resolve NB03 / NB04 / NB05 artifacts. Forgetting uses **OralXrays** metrics only
(never Tufts Dice from refit). McNemar prefers paired Stage-1/Stage-2 prob CSVs.


In [5]:
def _first_existing(paths):
    for p in paths:
        if os.path.isfile(p):
            return p
    return None

RESOLVED_INPUTS = {
    'stage1_metrics': _first_existing([
        'results/stage1_metrics.json',
    ]),
    # OralXrays forgetting ONLY — never stage2_refit_metrics.json (that file stores Tufts Dice).
    'stage2_oralxray_metrics': _first_existing([
        'results/stage2_oralxray_metrics.json',
        'results/stage2_metrics.json',
    ]),
    'stage1_tufts_preds': _first_existing([
        'results/stage1_tufts_zero_shot_predictions.csv',
        'results/stage1_tufts_per_image_predictions.csv',
    ]),
    # Prefer paired Stage-1 + Stage-2 probs on the same 100 held-out Tufts images.
    'stage2_tufts_preds': _first_existing([
        'results/stage2_refit_test_predictions.csv',
        'results/stage2_part1_test_predictions.csv',
        'results/stage2_tufts_per_image_predictions.csv',
    ]),
    # Optional: stored Youden thresholds from the same refit eval (avoids re-fitting on test).
    'stage2_refit_metrics': _first_existing([
        'results/stage2_refit_metrics.json',
    ]),
    'dentex_stage2_preds': _first_existing([
        'results/dentex_per_image_predictions.csv',
        'results/stage2_dentex_per_image_predictions.csv',
    ]),
    'dentex_baseline_preds': _first_existing([
        'results/dentex_baseline_predictions.csv',
    ]),
}

REQUIRED_FOR_REAL = [
    'stage1_metrics', 'stage2_oralxray_metrics',
    'stage1_tufts_preds', 'stage2_tufts_preds',
    'dentex_stage2_preds', 'dentex_baseline_preds',
]

all_present = True
for key, path in RESOLVED_INPUTS.items():
    need = key in REQUIRED_FOR_REAL
    tag = 'present' if path else ('MISSING' if need else 'optional')
    print(f'  [{tag:>8}] {key}: {path}')
    if need and path is None:
        all_present = False

print(f'All required inputs present: {all_present}')


  [ present] stage1_metrics: results/stage1_metrics.json
  [ present] stage2_oralxray_metrics: results/stage2_oralxray_metrics.json
  [ present] stage1_tufts_preds: results/stage1_tufts_zero_shot_predictions.csv
  [ present] stage2_tufts_preds: results/stage2_refit_test_predictions.csv
  [ present] stage2_refit_metrics: results/stage2_refit_metrics.json
  [ present] dentex_stage2_preds: results/dentex_per_image_predictions.csv
  [ present] dentex_baseline_preds: results/dentex_baseline_predictions.csv
All required inputs present: True


## 3 — Helper functions

- `run_forgetting_check` — OralXrays Dice ratio (§7.1)
- `build_paired_tufts_preds` — McNemar inputs with **stored** thresholds when available
- `run_mcnemar` — Tufts healthy pilot (§7.2)
- `run_dentex_ood_comparison` — AUROC + paired bootstrap CI (§7.3 reportable claim)


In [6]:
from core_modules import compute_dice, calibrate_decision_threshold
from negative_result_stats import bootstrap_ci, bootstrap_paired_diff_ci

# Prefer explicit OralXray keys. Generic 'Dice' is last-resort and only safe on files that
# are known to be OralXray evals (stage2_oralxray_metrics.json).
ORAL_DICE_KEYS = (
    'OralXray_forgetting_Dice',
    'OralXrays_Dice',
    'OralXray_test_Dice',
    'OralXray_Dice',
    'oralxray_Dice',
)


def _dig_metric(obj, keys):
    if not isinstance(obj, dict):
        return None
    for k in keys:
        v = obj.get(k)
        if isinstance(v, dict):
            v = v.get('mean', v.get('value'))
        if v is not None and not isinstance(v, dict):
            return float(v)
    for nest in ('oralxray_test_metrics', 'aggregate', 'metrics'):
        nested = obj.get(nest)
        if isinstance(nested, dict):
            found = _dig_metric(nested, keys)
            if found is not None:
                return found
    return None


def _oral_dice(metrics: dict, *, allow_generic_dice: bool = False) -> float:
    keys = ORAL_DICE_KEYS + (('Dice',) if allow_generic_dice else ())
    v = _dig_metric(metrics, keys)
    if v is None:
        raise KeyError(f'No OralXrays Dice key in metrics: {list(metrics)[:24]}')
    return v


def run_forgetting_check(stage1_metrics, stage2_metrics, *, stage2_path=None):
    """§7.1 — Forgetting ratio on OralXrays."""
    # stage2_oralxray_metrics.json is a flat OralXray eval → allow generic Dice there.
    allow = bool(stage2_path and os.path.basename(stage2_path) == 'stage2_oralxray_metrics.json')
    s1_dice = _oral_dice(stage1_metrics, allow_generic_dice=False)
    s2_dice = _oral_dice(stage2_metrics, allow_generic_dice=allow)
    ratio = (s1_dice - s2_dice) / s1_dice if s1_dice > 0 else float('nan')
    return {
        'stage1_OralXrays_Dice': s1_dice,
        'stage2_OralXrays_Dice': s2_dice,
        'forgetting_ratio': ratio,
        'stage2_source': stage2_path,
        'stage2_checkpoint': stage2_metrics.get('checkpoint'),
    }


STAGE2_PROB_COLS = ('prob_stage2_refit', 'prob_stage2_gaze', 'prob_stage2', 'probs')


def _load_stored_thresholds(refit_metrics_path):
    """Return (thr_stage1, thr_stage2) from refit metrics if present."""
    if not refit_metrics_path or not os.path.isfile(refit_metrics_path):
        return None, None
    with open(refit_metrics_path, 'r', encoding='utf-8') as f:
        m = json.load(f)
    thr1 = (m.get('stage1_test_metrics') or {}).get('opt_threshold')
    thr2 = (m.get('refit_test_metrics') or {}).get('opt_threshold')
    if thr1 is None or thr2 is None:
        return None, None
    return float(thr1), float(thr2)


def build_paired_tufts_preds(stage2_path, stage1_path, thr_stage1=None, thr_stage2=None):
    """
    Return (merged_df, meta) with has_lesion_gt / model_pred / baseline_pred.

    Threshold priority for paired-prob CSVs:
      1. stored Youden thresholds from stage2_refit_metrics.json (preferred)
      2. fallback: calibrate on this same held-out set (flagged in meta)
    """
    s2_df = pd.read_csv(stage2_path)
    cols = set(s2_df.columns)
    s2_prob_col = next((c for c in STAGE2_PROB_COLS if c in cols), None)

    if 'prob_stage1' in cols and s2_prob_col is not None:
        y = s2_df['has_lesion_gt'].astype(int).to_numpy()
        p2 = s2_df[s2_prob_col].to_numpy()
        p1 = s2_df['prob_stage1'].to_numpy()
        if thr_stage2 is None or thr_stage1 is None:
            thr2 = float(calibrate_decision_threshold(y, p2))
            thr1 = float(calibrate_decision_threshold(y, p1))
            thr_source = 'calibrated_on_heldout_test'
        else:
            thr2, thr1 = float(thr_stage2), float(thr_stage1)
            thr_source = 'stored_refit_metrics'
        merged = pd.DataFrame({
            'has_lesion_gt': y,
            'model_pred': (p2 > thr2).astype(int),
            'baseline_pred': (p1 > thr1).astype(int),
        })
        meta = {
            'layout': 'paired_probs',
            'stage2_file': stage2_path,
            'stage2_prob_col': s2_prob_col,
            'stage2_threshold': thr2,
            'stage1_threshold': thr1,
            'threshold_source': thr_source,
        }
        return merged, meta

    s1_df = pd.read_csv(stage1_path)
    for df in (s2_df, s1_df):
        if 'model_pred' not in df.columns and 'pred' in df.columns:
            df.rename(columns={'pred': 'model_pred'}, inplace=True)
        if 'id' not in df.columns and 'image_id' in df.columns:
            df.rename(columns={'image_id': 'id'}, inplace=True)

    if 'id' in s2_df.columns and 'id' in s1_df.columns:
        merged = s2_df[['id', 'has_lesion_gt', 'model_pred']].merge(
            s1_df[['id', 'model_pred']].rename(columns={'model_pred': 'baseline_pred'}),
            on='id', how='inner',
        )
        layout = 'binary_preds_merged_on_id'
    else:
        n = min(len(s2_df), len(s1_df))
        merged = s2_df.iloc[:n][['has_lesion_gt', 'model_pred']].copy()
        merged['baseline_pred'] = s1_df.iloc[:n]['model_pred'].to_numpy()
        layout = 'binary_preds_row_order'
        print('WARNING: merged Tufts preds by row order (no id column)')

    meta = {
        'layout': layout,
        'stage2_file': stage2_path,
        'stage1_file': stage1_path,
        'threshold_source': 'shipped_binary_preds',
    }
    return merged, meta


def run_mcnemar(predictions_df):
    """§7.2 — McNemar on Tufts healthy subset (exact, two-sided)."""
    from statsmodels.stats.contingency_tables import mcnemar

    healthy = predictions_df[predictions_df['has_lesion_gt'] == 0].copy()
    s2_ok = healthy['model_pred'].astype(int).to_numpy() == 0
    s1_ok = healthy['baseline_pred'].astype(int).to_numpy() == 0
    # Convention: [[both correct, S2 only], [S1 only, both wrong]]
    table = [
        [int((s2_ok & s1_ok).sum()), int((s2_ok & ~s1_ok).sum())],
        [int((~s2_ok & s1_ok).sum()), int((~s2_ok & ~s1_ok).sum())],
    ]
    result = mcnemar(table, exact=True)
    return result, table


def _prob_col(df):
    for c in ('probs', 'prob', 'prob_stage2', 'score'):
        if c in df.columns:
            return c
    raise KeyError(f'No probability column in {list(df.columns)}')


def _degenerate(pred):
    pred = np.asarray(pred)
    return bool(len(pred) > 0 and pred.min() == pred.max())


def run_dentex_ood_comparison(gaze_df, baseline_df, n_boot=2000, seed=42,
                              thr_stage1=None, thr_stage2=None):
    """
    §7.3 — DENTEX OOD comparison.

    Reportable claim = AUROC + paired bootstrap dAUROC.
    Thresholded correctness (Wilcoxon) uses *transferred* Tufts thresholds when provided;
    otherwise falls back to Youden on DENTEX itself (flagged). Shipped binary preds are
    diagnostic only (base-rate / degenerate thresholds).
    """
    from scipy.stats import wilcoxon

    y = gaze_df['has_lesion_gt'].astype(int).to_numpy()
    y_b = baseline_df['has_lesion_gt'].astype(int).to_numpy()
    n = min(len(y), len(y_b))
    y, y_b = y[:n], y_b[:n]
    if not np.array_equal(y, y_b):
        raise ValueError('DENTEX ground truth differs between Stage-2 and Stage-1 files')

    p2 = gaze_df[_prob_col(gaze_df)].to_numpy()[:n]
    p1 = baseline_df[_prob_col(baseline_df)].to_numpy()[:n]

    auroc2 = bootstrap_ci(y, p2, n_boot=n_boot, seed=seed)
    auroc1 = bootstrap_ci(y, p1, n_boot=n_boot, seed=seed)
    diff = bootstrap_paired_diff_ci(y, p1, p2, n_boot=n_boot, seed=seed)
    discrimination_significant = bool(diff['lower'] > 0 or diff['upper'] < 0)

    if thr_stage2 is None or thr_stage1 is None:
        thr2 = float(calibrate_decision_threshold(y, p2))
        thr1 = float(calibrate_decision_threshold(y, p1))
        thr_source = 'calibrated_on_dentex'
    else:
        thr2, thr1 = float(thr_stage2), float(thr_stage1)
        thr_source = 'transferred_from_tufts_refit'

    m2 = (p2 > thr2).astype(int)
    m1 = (p1 > thr1).astype(int)
    c2 = (m2 == y).astype(float)
    c1 = (m1 == y).astype(float)
    if np.allclose(c2, c1):
        w_stat, w_p = 0.0, 1.0
    else:
        w_stat, w_p = wilcoxon(c2, c1, alternative='greater')

    shipped2 = gaze_df['model_pred'].astype(int).to_numpy()[:n]
    shipped1 = baseline_df['model_pred'].astype(int).to_numpy()[:n]

    if diff['lower'] > 0:
        interpretation = 'Stage-2 discriminates better than Stage-1 on DENTEX'
    elif diff['upper'] < 0:
        interpretation = 'Stage-1 discriminates better than Stage-2 on DENTEX'
    else:
        interpretation = 'No discrimination difference on DENTEX (null OOD result)'

    return {
        'n_images': int(n),
        'prevalence': float(y.mean()),
        'stage2_auroc': auroc2,
        'stage1_auroc': auroc1,
        'paired_auroc_diff_stage2_minus_stage1': diff,
        'discrimination_significant': discrimination_significant,
        'matched_threshold_wilcoxon': {
            'statistic': float(w_stat),
            'p_value': float(w_p),
            'stage2_threshold': thr2,
            'stage1_threshold': thr1,
            'threshold_source': thr_source,
            'stage2_accuracy': float(c2.mean()),
            'stage1_accuracy': float(c1.mean()),
            'significant_at_0.05': bool(w_p < 0.05),
        },
        'shipped_threshold_diagnostics': {
            'stage2_positive_rate': float(shipped2.mean()),
            'stage1_positive_rate': float(shipped1.mean()),
            'stage2_accuracy': float((shipped2 == y).mean()),
            'stage1_accuracy': float((shipped1 == y).mean()),
            'stage2_degenerate': _degenerate(shipped2),
            'stage1_degenerate': _degenerate(shipped1),
            'warning': 'Accuracy at shipped thresholds is base-rate driven; not a discrimination claim.',
        },
        'interpretation': interpretation,
    }


print('Helpers OK: forgetting / paired Tufts / McNemar / DENTEX OOD')


Helpers OK: forgetting / paired Tufts / McNemar / DENTEX OOD


## 4 — Pre-flight code check (always runs)


In [7]:
CODE_CHECK_PASSED = False

synth_s1 = {'OralXray_test_Dice': 0.80}
synth_s2 = {'OralXray_forgetting_Dice': 0.72}
forgetting = run_forgetting_check(synth_s1, synth_s2)
assert np.isfinite(forgetting['forgetting_ratio'])
print(f'Pre-flight forgetting: ratio={forgetting["forgetting_ratio"]:.2%}')

# Guard: Tufts-only metrics must NOT be accepted as OralXray Dice
try:
    run_forgetting_check({'OralXray_test_Dice': 0.9}, {'Dice': 0.4, 'role': 'secondary_refit_eval'})
    raise AssertionError('refit Tufts Dice was incorrectly accepted as OralXray Dice')
except KeyError:
    print('Pre-flight guard: refused generic Dice on non-OralXray metrics')

synth_preds = pd.DataFrame({
    'has_lesion_gt': [0] * 10,
    'model_pred':    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    'baseline_pred': [0, 1, 1, 0, 0, 1, 0, 1, 0, 1],
})
mc_result, mc_table = run_mcnemar(synth_preds)
assert len(mc_table) == 2 and all(len(r) == 2 for r in mc_table)
assert np.isfinite(mc_result.pvalue)
print(f'Pre-flight McNemar: p={mc_result.pvalue:.4f} | table={mc_table}')

rng = np.random.default_rng(0)
_y = rng.integers(0, 2, 120)
synth_ood_gaze = pd.DataFrame({
    'has_lesion_gt': _y,
    'model_pred': np.ones(120, dtype=int),
    'probs': np.clip(0.6 + 0.1 * _y + rng.normal(0, 0.05, 120), 0, 1),
})
synth_ood_base = pd.DataFrame({
    'has_lesion_gt': _y,
    'model_pred': np.zeros(120, dtype=int),
    'probs': np.clip(0.2 + 0.1 * _y + rng.normal(0, 0.05, 120), 0, 1),
})
ood_check = run_dentex_ood_comparison(synth_ood_gaze, synth_ood_base, n_boot=100, seed=0)
assert np.isfinite(ood_check['paired_auroc_diff_stage2_minus_stage1']['mean_diff'])
assert ood_check['shipped_threshold_diagnostics']['stage1_degenerate'] is True
print(f"Pre-flight DENTEX OOD: dAUROC={ood_check['paired_auroc_diff_stage2_minus_stage1']['mean_diff']:+.4f} "
      f"| {ood_check['interpretation']}")

CODE_CHECK_PASSED = True
print('CODE CHECK PASSED')


Pre-flight forgetting: ratio=10.00%
Pre-flight guard: refused generic Dice on non-OralXray metrics
Pre-flight McNemar: p=0.4531 | table=[[3, 5], [2, 0]]
Pre-flight DENTEX OOD: dAUROC=+0.0179 | No discrimination difference on DENTEX (null OOD result)
CODE CHECK PASSED


## 5 — Real statistical tests

Runs whenever required inputs are present (CPU or GPU). Uses stored Tufts Youden
thresholds from `stage2_refit_metrics.json` when available.


In [ ]:
REAL_STATS_DONE = False
mcnemar_result = None
mcnemar_table = None
forgetting_info = None
tufts_pair_meta = None
dentex_ood = None
dentex_ood_hardened = None
wilcoxon_stat = None
wilcoxon_pval = None
wilcoxon_gaze_dice = None
wilcoxon_baseline_dice = None

N_BOOT = int(CONFIG.get('negative_result_bootstrap_n', 2000))
thr1_stored, thr2_stored = _load_stored_thresholds(RESOLVED_INPUTS.get('stage2_refit_metrics'))
print(f'Stored Tufts thresholds: stage1={thr1_stored} | stage2={thr2_stored}')

if not all_present:
    print('SKIPPING real statistical tests — required upstream inputs are missing.')
else:
    with open(RESOLVED_INPUTS['stage1_metrics'], 'r', encoding='utf-8') as f:
        stage1_metrics = json.load(f)
    with open(RESOLVED_INPUTS['stage2_oralxray_metrics'], 'r', encoding='utf-8') as f:
        stage2_oral_metrics = json.load(f)

    forgetting_info = run_forgetting_check(
        stage1_metrics, stage2_oral_metrics,
        stage2_path=RESOLVED_INPUTS['stage2_oralxray_metrics'],
    )
    print(f"Stage-1 OralXrays Dice: {forgetting_info['stage1_OralXrays_Dice']:.4f}")
    print(f"Stage-2 OralXrays Dice: {forgetting_info['stage2_OralXrays_Dice']:.4f}")
    print(f"Forgetting Ratio: {forgetting_info['forgetting_ratio']:.2%}")
    print(f"  source={forgetting_info['stage2_source']} | ckpt={forgetting_info.get('stage2_checkpoint')}")

    merged, tufts_pair_meta = build_paired_tufts_preds(
        RESOLVED_INPUTS['stage2_tufts_preds'],
        RESOLVED_INPUTS['stage1_tufts_preds'],
        thr_stage1=thr1_stored,
        thr_stage2=thr2_stored,
    )
    print('\nTufts pairing:', tufts_pair_meta)
    n_healthy = int((merged['has_lesion_gt'] == 0).sum())
    print(f'McNemar: {len(merged)} images, {n_healthy} healthy (pilot / underpowered)')
    mcnemar_result, mcnemar_table = run_mcnemar(merged)
    print(f'McNemar p-value: {mcnemar_result.pvalue:.4f} | table={mcnemar_table}')

    gaze_df = pd.read_csv(RESOLVED_INPUTS['dentex_stage2_preds'])
    baseline_df = pd.read_csv(RESOLVED_INPUTS['dentex_baseline_preds'])

    dentex_ood = run_dentex_ood_comparison(
        gaze_df, baseline_df, n_boot=N_BOOT,
        thr_stage1=thr1_stored, thr_stage2=thr2_stored,
    )
    d = dentex_ood['paired_auroc_diff_stage2_minus_stage1']
    mw = dentex_ood['matched_threshold_wilcoxon']
    print(f"\nDENTEX OOD (n={dentex_ood['n_images']}, prevalence={dentex_ood['prevalence']:.2%})")
    print(f"  Stage-2 AUROC {dentex_ood['stage2_auroc']['point_estimate']:.4f} "
          f"[{dentex_ood['stage2_auroc']['lower']:.4f}, {dentex_ood['stage2_auroc']['upper']:.4f}]")
    print(f"  Stage-1 AUROC {dentex_ood['stage1_auroc']['point_estimate']:.4f} "
          f"[{dentex_ood['stage1_auroc']['lower']:.4f}, {dentex_ood['stage1_auroc']['upper']:.4f}]")
    print(f"  Paired dAUROC {d['mean_diff']:+.4f} [{d['lower']:+.4f}, {d['upper']:+.4f}] "
          f"-> {dentex_ood['interpretation']}")
    print(f"  Thresholded Wilcoxon ({mw['threshold_source']}): W={mw['statistic']:.1f}, "
          f"p={mw['p_value']:.4g}, acc S2={mw['stage2_accuracy']:.3f} S1={mw['stage1_accuracy']:.3f}")

    diag = dentex_ood['shipped_threshold_diagnostics']
    if diag['stage1_degenerate'] or diag['stage2_degenerate']:
        print(f"  WARNING: degenerate shipped preds "
              f"(S2 pos rate {diag['stage2_positive_rate']:.3f}, "
              f"S1 {diag['stage1_positive_rate']:.3f}) — not reportable.")

    # Shipped-threshold Wilcoxon kept only as a diagnostic trace.
    from scipy.stats import wilcoxon as _wilcoxon
    y = gaze_df['has_lesion_gt'].astype(int).to_numpy()
    c2 = (gaze_df['model_pred'].astype(int).to_numpy() == y).astype(float)
    c1 = (baseline_df['model_pred'].astype(int).to_numpy()[:len(c2)] == y[:len(c2)]).astype(float)
    if np.allclose(c2, c1):
        wilcoxon_stat, wilcoxon_pval = 0.0, 1.0
    else:
        wilcoxon_stat, wilcoxon_pval = _wilcoxon(c2, c1, alternative='greater')
    wilcoxon_gaze_dice = float(compute_dice(gaze_df['model_pred'].to_numpy(), y))
    wilcoxon_baseline_dice = float(compute_dice(baseline_df['model_pred'].to_numpy()[:len(y)], y))
    print(f"  [shipped Wilcoxon DIAGNOSTIC ONLY] W={wilcoxon_stat:.1f}, p={wilcoxon_pval:.4g}, "
          f"Dice {wilcoxon_gaze_dice:.4f} vs {wilcoxon_baseline_dice:.4f}")

    # DENTEX hardening: AP + balanced-subsample AUROC (no re-inference)
    from ood_hardening import harden_dentex_ood
    dentex_ood_hardened = harden_dentex_ood(
        RESOLVED_INPUTS['dentex_stage2_preds'],
        RESOLVED_INPUTS['dentex_baseline_preds'],
        n_boot=N_BOOT,
        n_balanced_draws=1000,
    )
    os.makedirs('statistics', exist_ok=True)
    with open('statistics/dentex_ood_hardened.json', 'w', encoding='utf-8') as f:
        json.dump(dentex_ood_hardened, f, indent=2)
    bal = dentex_ood_hardened['balanced_subsample_auroc']
    print('\nDENTEX hardened OOD:')
    ap2 = dentex_ood_hardened['average_precision']['stage2']['point_estimate']
    ap1 = dentex_ood_hardened['average_precision']['stage1']['point_estimate']
    print(f"  AP S2={ap2:.4f} S1={ap1:.4f} (random~={dentex_ood_hardened['random_ap_baseline']:.3f})")
    if bal.get('ok'):
        d = bal['paired_auroc_diff_stage2_minus_stage1_balanced']
        print(f"  Balanced dAUROC={d['mean']:+.4f} [{d['lower']:+.4f}, {d['upper']:+.4f}] -> {bal['interpretation']}")

    REAL_STATS_DONE = True
    print('\nREAL STATISTICAL ANALYSIS COMPLETE.')


## 6 — Save artifacts

Writes `statistics/mcnemar_table.csv` and `statistics/p_values.json` when real tests ran.


In [ ]:
if REAL_STATS_DONE:
    pd.DataFrame(mcnemar_table).to_csv('statistics/mcnemar_table.csv', index=False)
    print('Saved statistics/mcnemar_table.csv')

    d = dentex_ood['paired_auroc_diff_stage2_minus_stage1']
    p_values = {
        'dentex_ood_discrimination': {
            'stage2_auroc': dentex_ood['stage2_auroc'],
            'stage1_auroc': dentex_ood['stage1_auroc'],
            'paired_auroc_diff_stage2_minus_stage1': d,
            'significant_at_0.05': dentex_ood['discrimination_significant'],
            'interpretation': dentex_ood['interpretation'],
            'n_images': dentex_ood['n_images'],
            'prevalence': dentex_ood['prevalence'],
        },
        'wilcoxon_H1_gaze_vs_baseline_DENTEX': {
            **dentex_ood['matched_threshold_wilcoxon'],
            'basis': 'per-image correctness at transferred Tufts Youden thresholds (fallback: DENTEX Youden)',
        },
        'wilcoxon_shipped_threshold_DENTEX_diagnostic_only': {
            'statistic': float(wilcoxon_stat),
            'p_value': float(wilcoxon_pval),
            'gaze_dice': float(wilcoxon_gaze_dice),
            'baseline_dice': float(wilcoxon_baseline_dice),
            **dentex_ood['shipped_threshold_diagnostics'],
        },
        'mcnemar_tufts_healthy_subset': {
            'p_value': float(mcnemar_result.pvalue),
            'significant_at_0.05': bool(mcnemar_result.pvalue < 0.05),
            'table': mcnemar_table,
            'pairing': tufts_pair_meta,
            'note': 'Pilot / underpowered (n_healthy ~66). Primary detection claim is NB04 CV.',
        },
        'forgetting_ratio': {
            'stage1_OralXrays_Dice': float(forgetting_info['stage1_OralXrays_Dice']),
            'stage2_OralXrays_Dice': float(forgetting_info['stage2_OralXrays_Dice']),
            'forgetting_ratio': float(forgetting_info['forgetting_ratio']),
            'stage2_source': forgetting_info.get('stage2_source'),
            'stage2_checkpoint': forgetting_info.get('stage2_checkpoint'),
        },
        'inputs': RESOLVED_INPUTS,
        'n_boot': N_BOOT,
        'dentex_ood_hardened': dentex_ood_hardened,
        'note': (
            'Secondary confirmatory tests. Primary detection claim = NB04 CV + bootstrap. '
            'DENTEX reportable claim = AUROC discrimination (not shipped-threshold accuracy).'
        ),
    }
    with open('statistics/p_values.json', 'w', encoding='utf-8') as f:
        json.dump(p_values, f, indent=4)
    print('Saved statistics/p_values.json')
    print(json.dumps(p_values, indent=2))
else:
    print('No artifacts written — inputs missing or real tests skipped.')


## 7 — Save run status (always written)


In [10]:
status = {
    'mode': 'gpu' if ON_GPU else 'cpu',
    'gpu_required': False,
    'real_stats_done': bool(REAL_STATS_DONE),
    'code_check_passed': bool(globals().get('CODE_CHECK_PASSED', False)),
    'all_inputs_present': bool(all_present),
    'inputs': RESOLVED_INPUTS,
    'tufts_pairing': tufts_pair_meta,
    'dentex_interpretation': None if not dentex_ood else dentex_ood['interpretation'],
    'forgetting_ratio': None if not forgetting_info else forgetting_info.get('forgetting_ratio'),
}
with open('statistics/statistics_status.json', 'w', encoding='utf-8') as f:
    json.dump(status, f, indent=2)
print('Saved statistics/statistics_status.json')
print(json.dumps(status, indent=2))


Saved statistics/statistics_status.json
{
  "mode": "cpu",
  "gpu_required": false,
  "real_stats_done": true,
  "code_check_passed": true,
  "all_inputs_present": true,
  "inputs": {
    "stage1_metrics": "results/stage1_metrics.json",
    "stage2_oralxray_metrics": "results/stage2_oralxray_metrics.json",
    "stage1_tufts_preds": "results/stage1_tufts_zero_shot_predictions.csv",
    "stage2_tufts_preds": "results/stage2_refit_test_predictions.csv",
    "stage2_refit_metrics": "results/stage2_refit_metrics.json",
    "dentex_stage2_preds": "results/dentex_per_image_predictions.csv",
    "dentex_baseline_preds": "results/dentex_baseline_predictions.csv"
  },
  "tufts_pairing": {
    "layout": "paired_probs",
    "stage2_file": "results/stage2_refit_test_predictions.csv",
    "stage2_prob_col": "prob_stage2_refit",
    "stage2_threshold": 0.7939968109130859,
    "stage1_threshold": 0.0962478294968605,
    "threshold_source": "stored_refit_metrics"
  },
  "dentex_interpretation": "No dis

## 8 — Summary table


In [11]:
if REAL_STATS_DONE:
    d = dentex_ood['paired_auroc_diff_stage2_minus_stage1']
    mw = dentex_ood['matched_threshold_wilcoxon']
    print()
    print('=' * 72)
    print(f'  {"Test":<34} {"Result":>16}  {"p / CI":>14}  {"Sig?":>5}')
    print('-' * 72)
    print(f'  {"Forgetting ratio (OralXrays)":<34} {forgetting_info["forgetting_ratio"]:>15.2%}  {"N/A":>14}  {"N/A":>5}')
    print(f'  {"McNemar (Tufts healthy)":<34} {str(mcnemar_table):>16}  {mcnemar_result.pvalue:>14.4f}  '
          f'{"YES" if mcnemar_result.pvalue < 0.05 else "no":>5}')
    ci_txt = '[{:+.3f},{:+.3f}]'.format(d['lower'], d['upper'])
    print(f'  {"DENTEX dAUROC (S2 - S1)":<34} {d["mean_diff"]:>+16.4f}  {ci_txt:>14}  '
          f'{"YES" if dentex_ood["discrimination_significant"] else "no":>5}')
    print(f'  {"DENTEX Wilcoxon (thr)":<34} {mw["statistic"]:>16.0f}  {mw["p_value"]:>14.4g}  '
          f'{"YES" if mw["significant_at_0.05"] else "no":>5}')
    print('=' * 72)
    print(f'  DENTEX: {dentex_ood["interpretation"]}')
    print(f'  thresholds: {mw["threshold_source"]}')
    print('=' * 72)
else:
    print('No summary table — inputs missing.')



  Test                                         Result          p / CI   Sig?
------------------------------------------------------------------------
  Forgetting ratio (OralXrays)                 3.96%             N/A    N/A
  McNemar (Tufts healthy)            [[51, 5], [7, 3]]          0.7744     no
  DENTEX dAUROC (S2 - S1)                     -0.0377  [-0.129,+0.049]     no
  DENTEX Wilcoxon (thr)                         12764               1     no
  DENTEX: No discrimination difference on DENTEX (null OOD result)
  thresholds: transferred_from_tufts_refit


## 9 — Final status banner


In [12]:
from IPython.display import Markdown, display

if REAL_STATS_DONE:
    d = dentex_ood['paired_auroc_diff_stage2_minus_stage1']
    mw = dentex_ood['matched_threshold_wilcoxon']
    banner = f"""
# NB07 COMPLETE — SECONDARY STATISTICAL TESTS

| Test | Result | p / CI | Significant? |
|---|---|---|---|
| **Forgetting ratio** (OralXrays) | **{forgetting_info['forgetting_ratio']:.2%}** | N/A | N/A |
| **McNemar** (Tufts healthy) | table={mcnemar_table} | p={mcnemar_result.pvalue:.4f} | {'YES' if mcnemar_result.pvalue < 0.05 else 'no'} |
| **DENTEX dAUROC** (S2 - S1) | {d['mean_diff']:+.4f} | [{d['lower']:+.4f}, {d['upper']:+.4f}] | {'YES' if dentex_ood['discrimination_significant'] else 'no'} |
| **DENTEX Wilcoxon** (thresholded) | W={mw['statistic']:.0f} | p={mw['p_value']:.4g} | {'YES' if mw['significant_at_0.05'] else 'no'} |

**DENTEX:** {dentex_ood['interpretation']}
**Threshold source:** `{mw['threshold_source']}`

Primary detection claim stays **NB04 CV + bootstrap**. Shipped-threshold DENTEX accuracy is diagnostic only.

**Saved:** `statistics/mcnemar_table.csv`, `statistics/p_values.json`.
"""
    box = '#' * 64
    print(box)
    print('#  NB07 STATISTICAL ANALYSIS COMPLETED')
    print('#  Forgetting: {:.2%} | McNemar p={:.4f} | DENTEX dAUROC={:+.4f}'.format(
        forgetting_info['forgetting_ratio'], mcnemar_result.pvalue, d['mean_diff']))
    print(box)
else:
    banner = f"""
# TEST-ONLY RUN — NO REAL STATS

| | |
|---|---|
| **Code check** | {'PASSED' if status.get('code_check_passed') else 'FAILED'} |
| **Real tests** | **NOT performed** |

Produce NB03/NB04/NB05 artifacts, then re-run — no GPU required.
"""
    box = '!' * 64
    print(box)
    print('!!  TEST-ONLY RUN — NO STATISTICAL ANALYSIS WAS PERFORMED  !!')
    print(box)

display(Markdown(banner))


################################################################
#  NB07 STATISTICAL ANALYSIS COMPLETED
#  Forgetting: 3.96% | McNemar p=0.7744 | DENTEX dAUROC=-0.0377
################################################################



# NB07 COMPLETE — SECONDARY STATISTICAL TESTS

| Test | Result | p / CI | Significant? |
|---|---|---|---|
| **Forgetting ratio** (OralXrays) | **3.96%** | N/A | N/A |
| **McNemar** (Tufts healthy) | table=[[51, 5], [7, 3]] | p=0.7744 | no |
| **DENTEX dAUROC** (S2 - S1) | -0.0377 | [-0.1286, +0.0489] | no |
| **DENTEX Wilcoxon** (thresholded) | W=12764 | p=1 | no |

**DENTEX:** No discrimination difference on DENTEX (null OOD result)  
**Threshold source:** `transferred_from_tufts_refit`

Primary detection claim stays **NB04 CV + bootstrap**. Shipped-threshold DENTEX accuracy is diagnostic only.

**Saved:** `statistics/mcnemar_table.csv`, `statistics/p_values.json`.
